# K6 · Start the ResearchLens app (Streamlit + temporary public link)
Needs: **Internet on** and the secrets `HF_TOKEN` + `GEMINI_API_KEY`. **No GPU needed** (Accelerator: None):
our models run on the CPU and answers come from the free Gemini API. Takes about 3 min to start; the first question
loads the models (about 1 min). The link works while this notebook session runs — stop the session when you are done
.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## Gemini key test (about 10 s)
Add-ons → Secrets → add **`GEMINI_API_KEY`** (free key from https://aistudio.google.com/apikey) and tick it for this
notebook. The app sends the retrieved evidence (text + figure/table images) to Gemini, which writes the answer with
[E#] citations. The key is never printed.

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY"); print("GEMINI_API_KEY loaded")
from PIL import Image, ImageDraw
from researchlens.pipeline import answer as A
print("will try:", A.gemini_models())
img = Image.new("RGB", (320, 120), "white"); ImageDraw.Draw(img).text((20, 45), "Table 1: Swin-B top-1 = 86.4", fill="black")
ev = [{"eid": "E1", "kind": "table", "label": "Table 1", "image": None,
       "text": "Table 1 (ImageNet-1K). Swin-B, 384^2, ImageNet-22K pre-trained: top-1 86.4", "paper_id": "test",
       "page": 1, "block_id": "t1", "via": "test"}]
msgs = A.build_messages(ev, "What top-1 accuracy does Swin-B reach? Also say what text you can read in the image.")
msgs[1]["content"][:0] = [{"type": "text", "text": "Image of [E1] (Table 1):"}, {"type": "image", "image": img}]
out = A.generate(None, msgs, max_new_tokens=200, on_text=lambda t: None)
print("answered by:", A.LAST_MODEL["name"])
print("answer:", out)
print("Evidence Trace:", [t["eid"] for t in A.evidence_trace(out, ev)])

In [ ]:
!pip install -q -r requirements-app.txt
import subprocess, re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
os.environ["RL_WORK_DIR"] = "/kaggle/working/rl_app"
app = subprocess.Popen("streamlit run app/streamlit_app.py --server.port 8501 --server.headless true", shell=True,
                       stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
tun = subprocess.Popen("/tmp/cloudflared tunnel --url http://localhost:8501 --no-autoupdate", shell=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("ResearchLens app:", m.group(0)); break

## Update the app to the latest code from GitHub (without restarting the session)

In [ ]:
print(subprocess.run("git -C /tmp/rl/repo pull 2>&1; git -C /tmp/rl/repo log --oneline -1", shell=True, capture_output=True, text=True).stdout)
subprocess.run(["pkill", "-f", "streamlit"]); import time; time.sleep(3)
app = subprocess.Popen("streamlit run app/streamlit_app.py --server.port 8501 --server.headless true", shell=True,
                       stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
print("app restarted - reload the page and upload the paper again")

## App log (if something looks wrong)

In [ ]:
!tail -n 40 /tmp/streamlit.log